### This script is developed to extract rainfall and assing to every point the amount of rainfall the deformation recording day and 10 previous days to it

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import ee

In [2]:

ee.Authenticate()


True

In [3]:
ee.Initialize()


In [30]:
beolloeste = pd.read_csv(r"C:\Users\julia\MGEO\THESIS\DEFORMATION_DATA\culstered_landslides_forNN\SanCristobalNorte2_forNN.csv")

In [5]:
beolloeste

,X,Y,t_datetime,t_days,u,X_norm,Y_norm,t_norm,u_norm
0,4.705589e+06,2.251914e+06,2022-02-23,0.0,0.000000,0.300879,1.000000,0.0,0.000000
1,4.705469e+06,2.251874e+06,2022-02-23,0.0,0.000000,0.000382,0.901145,0.0,0.000000
2,4.705509e+06,2.251874e+06,2022-02-23,0.0,0.000000,0.100420,0.900764,0.0,0.000000
3,4.705549e+06,2.251874e+06,2022-02-23,0.0,0.000000,0.200458,0.900382,0.0,0.000000
4,4.705589e+06,2.251873e+06,2022-02-23,0.0,0.000000,0.300497,0.900000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...
3765,4.705628e+06,2.251593e+06,2024-04-01,768.0,0.000695,0.397862,0.199618,1.0,0.011654
3766,4.705668e+06,2.251593e+06,2024-04-01,768.0,-0.000850,0.497900,0.199237,1.0,-0.014237
3767,4.705508e+06,2.251514e+06,2024-04-01,768.0,0.003575,0.096984,0.000763,1.0,0.059901
3768,4.705548e+06,2.251513e+06,2024-04-01,768.0,0.005564,0.197022,0.000382,1.0,0.093236


In [31]:
beolloeste["t_datetime"] = pd.to_datetime(beolloeste["t_datetime"])

gdf = gpd.GeoDataFrame(
    beolloeste,
    geometry=gpd.points_from_xy(beolloeste["X"], beolloeste["Y"]),
    crs="EPSG:9377"  # <-- CAMBIA ESTO al EPSG correcto de tus X,Y
)

gdf_wgs = gdf.to_crs("EPSG:4326")
gdf_wgs["lon"] = gdf_wgs.geometry.x
gdf_wgs["lat"] = gdf_wgs.geometry.y

# nos quedamos con lo necesario
df_pts = gdf_wgs.drop(columns="geometry").copy()
df_pts.head()


,X,Y,t_datetime,t_days,u,X_norm,Y_norm,t_norm,u_norm,lon,lat
0,4.707363e+06,2.255309e+06,2022-02-23,0.0,0.0,0.889950,1.000000,0.0,0.0,-75.645879,6.303875
1,4.707323e+06,2.255269e+06,2022-02-23,0.0,0.0,0.834395,0.958407,0.0,0.0,-75.646240,6.303513
2,4.707363e+06,2.255269e+06,2022-02-23,0.0,0.0,0.889738,0.958247,0.0,0.0,-75.645879,6.303513
3,4.707403e+06,2.255269e+06,2022-02-23,0.0,0.0,0.945081,0.958087,0.0,0.0,-75.645517,6.303514
4,4.707283e+06,2.255229e+06,2022-02-23,0.0,0.0,0.778839,0.916814,0.0,0.0,-75.646602,6.303151


In [32]:


chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

def make_lag_stack(date_ee, lags=10):
    # Devuelve una imagen con 11 bandas: p_lag0, p_lag1, ... p_lag10
    imgs = []
    for k in range(lags + 1):
        d = date_ee.advance(-k, "day")
        img_k = chirps.filterDate(d, d.advance(1, "day")).first()
        # renombrar banda
        img_k = ee.Image(img_k).rename(f"p_lag{k}")
        imgs.append(img_k)
    return ee.Image.cat(imgs)


In [33]:
def df_to_fc(df_sub):
    feats = []
    for _, r in df_sub.iterrows():
        feats.append(
            ee.Feature(
                ee.Geometry.Point([float(r["lon"]), float(r["lat"])]),
                {
                    "row_id": int(r["row_id"]),
                    "t_datetime": r["t_datetime"].strftime("%Y-%m-%d")
                }
            )
        )
    return ee.FeatureCollection(feats)

# agrega un id para poder mergear bien al final
df_pts = df_pts.reset_index(drop=True)
df_pts["row_id"] = np.arange(len(df_pts))

unique_dates = sorted(df_pts["t_datetime"].dt.date.unique())

results = []

for d in unique_dates:
    df_sub = df_pts[df_pts["t_datetime"].dt.date == d].copy()
    fc = df_to_fc(df_sub)

    date_ee = ee.Date(str(d))
    stack = make_lag_stack(date_ee, lags=90)

    sampled = stack.sampleRegions(
        collection=fc,
        scale=5000,        # CHIRPS es ~5.5km, 5000 va bien
        geometries=False
    )

    # trae a pandas (ojo: si una fecha tiene miles de puntos, puede tardar)
    sampled_info = sampled.getInfo()
    feats = sampled_info["features"]

    for f in feats:
        props = f["properties"]
        results.append(props)

df_prec = pd.DataFrame(results)
df_prec.head()


,p_lag0,p_lag1,p_lag10,p_lag11,p_lag12,p_lag13,p_lag14,p_lag15,p_lag16,p_lag17,...,p_lag84,p_lag85,p_lag86,p_lag87,p_lag88,p_lag89,p_lag9,p_lag90,row_id,t_datetime
0,0.0,0.0,8.463918,6.416828,21.955637,0.0,0.0,0.0,5.424274,5.074764,...,3.830159,14.297736,7.148868,0.0,0.0,7.148868,27.972645,30.159897,0,2022-02-23
1,0.0,0.0,8.463918,6.416828,21.955637,0.0,0.0,0.0,5.424274,5.074764,...,3.830159,14.297736,7.148868,0.0,0.0,7.148868,27.972645,30.159897,1,2022-02-23
2,0.0,0.0,8.463918,6.416828,21.955637,0.0,0.0,0.0,5.424274,5.074764,...,3.830159,14.297736,7.148868,0.0,0.0,7.148868,27.972645,30.159897,2,2022-02-23
3,0.0,0.0,8.463918,6.416828,21.955637,0.0,0.0,0.0,5.424274,5.074764,...,3.830159,14.297736,7.148868,0.0,0.0,7.148868,27.972645,30.159897,3,2022-02-23
4,0.0,0.0,8.463918,6.416828,21.955637,0.0,0.0,0.0,5.424274,5.074764,...,3.830159,14.297736,7.148868,0.0,0.0,7.148868,27.972645,30.159897,4,2022-02-23


In [34]:
final = df_pts.merge(df_prec, on=["row_id"], how="left")

# ya tienes columnas p_lag0 ... p_lag10
final[["p_lag0","p_lag1","p_lag2","p_lag3","p_lag4","p_lag5","p_lag6","p_lag7","p_lag8","p_lag9","p_lag10"]].head()


,p_lag0,p_lag1,p_lag2,p_lag3,p_lag4,p_lag5,p_lag6,p_lag7,p_lag8,p_lag9,p_lag10
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.637923,27.972645,8.463918
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.637923,27.972645,8.463918
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.637923,27.972645,8.463918
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.637923,27.972645,8.463918
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.637923,27.972645,8.463918


In [35]:
print(final.columns)

Index(['X', 'Y', 't_datetime_x', 't_days', 'u', 'X_norm', 'Y_norm', 't_norm',
       'u_norm', 'lon',
       ...
       'p_lag83', 'p_lag84', 'p_lag85', 'p_lag86', 'p_lag87', 'p_lag88',
       'p_lag89', 'p_lag9', 'p_lag90', 't_datetime_y'],
      dtype='object', length=104)


In [36]:
# columnas de lags
lags_30 = [f"p_lag{i}" for i in range(1, 31)]
lags_60 = [f"p_lag{i}" for i in range(1, 61)]
lags_90 = [f"p_lag{i}" for i in range(1, 91)]

# acumulados
final["p_acum_30d"] = final[lags_30].sum(axis=1)
final["p_acum_60d"] = final[lags_60].sum(axis=1)
final["p_acum_90d"] = final[lags_90].sum(axis=1)


In [37]:
df_final = final[['X', 'Y', 't_datetime_x', 't_days', 'u', "p_acum_30d", "p_acum_60d", "p_acum_90d"]]

In [38]:
df_final.to_csv(r"C:\Users\julia\MGEO\THESIS\DEFORMATION_DATA\culstered_landslides_forNN_withRainfall\SanCristobalNorte2_with_rainfall.csv")